# Watermark Anything: four inpainting methods

Repository: [https://github.com/facebookresearch/watermark-anything](https://github.com/facebookresearch/watermark-anything)

Paper: [https://arxiv.org/abs/2411.07231](https://arxiv.org/abs/2411.07231)

Inspected commit: `2c08af04d037d5667c02f6ddebbda9ff04581c3e`

Watermark stage: `post_hoc`
Domain: image (framewise video adapter)

This method can be applied to existing media, so the notebook uses the fixed MOSE sample.

ProPainter, DiffuEraser, EffectErase, and E2FGVI-HQ receive the same authentic watermarked frames and binary mask. One official DiffuEraser execution is used: its internal ProPainter prior is preserved as the ProPainter result, and its final refinement is the DiffuEraser result. Inputs shorter than 22 frames are padded by repeating only the last frame/mask to 24 for that pipeline, then both reported outputs are trimmed to the authentic length. EffectErase applies its released Wan 4n+1 temporal adapter; E2FGVI-HQ uses the authors' official arbitrary-resolution test entry point.

The official `wam_mit.pth` model embeds one fixed 32-bit payload into every MOSE frame. WAM's native watermark-presence map is inverted at threshold 0.5 to score the removed region; that definition is recorded in the detector JSON.


## Authenticity contract

- Watermark preparation: Apply the official WAM embedder independently to every video frame using one fixed message, then reassemble the frames at the source frame rate.
- Checkpoint status: READY: the official MIT-licensed wam_mit.pth checkpoint is published by Meta.
- Protocol note: The paper is image-native.  The adapter must use the same message and official preprocessing for every frame and must record that this is a framewise video evaluation.
- No result cell fabricates metrics. Native extraction must finish and write its JSON before values are displayed.
- A method without a released spatial detector reports bit recovery only; it is not assigned a proxy localizer.


In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys
from IPython.display import JSON, Video, display

def find_repo_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "scripts/remaining_watermark_methods.py").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter inside the paper-reproductions-task1 checkout")

ROOT = find_repo_root()
SCRIPTS = ROOT / "scripts"

def run(command, *, cwd=None, env=None):
    command = [str(value) for value in command]
    print("+", shlex.join(command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

def require(path, description):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")
    return path

def require_commit(repository, expected):
    actual = subprocess.check_output(["git", "-C", str(repository), "rev-parse", "HEAD"], text=True).strip()
    if actual != expected:
        raise RuntimeError(f"Expected {expected} in {repository}, found {actual}")

def show_results(run_root):
    run_root = Path(run_root)
    for name in ("watermarked.mp4", "aligned-mask.mp4"):
        path = run_root / name
        if path.is_file():
            print(name)
            display(Video(str(path), embed=True, html_attributes="controls muted loop"))
    attack = run_root / "attack"
    for name in (
        "input-watermarked.mp4", "gt-mask.mp4", "propainter.mp4", "diffueraser.mp4",
        "effecterase.mp4", "e2fgvi.mp4",
    ):
        path = attack / name
        if path.is_file():
            print(name)
            display(Video(str(path), embed=True, html_attributes="controls muted loop"))
    result_json = {
        *run_root.rglob("summary.json"),
        *(attack / name for name in ("sample.json", "removal-pair.json", "effecterase.json", "e2fgvi.json")),
    }
    for path in sorted(path for path in result_json if path.is_file()):
        print(path.relative_to(run_root))
        display(JSON(json.loads(path.read_text())))

print("Repository:", ROOT)


In [ ]:
WAM = Path("/path/to/watermark-anything")
WAM_PYTHON = WAM / ".venv/bin/python"
WAM_CHECKPOINT = WAM / "checkpoints/wam_mit.pth"
WAM_PARAMS = WAM / "checkpoints/params.json"
DIFFUERASER = Path("/path/to/DiffuEraser")
DIFFUERASER_PYTHON = DIFFUERASER / ".venv/bin/python"
GPU = "0"
MOSE_VIDEO = ROOT / "evidence/mose-task1/review50/media/02b72b8d/original.mp4"
MOSE_MASK = ROOT / "evidence/mose-task1/review50/media/02b72b8d/gt_mask.mp4"
RUN_ROOT = ROOT / "runs/remaining-watermarks/watermark-anything/02b72b8d"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
WATERMARKED_VIDEO = RUN_ROOT / "watermarked.mp4"
ALIGNED_MASK = MOSE_MASK


Download the official checkpoint with `wget https://dl.fbaipublicfiles.com/watermark_anything/wam_mit.pth -P checkpoints/` from the WAM checkout.


In [ ]:
require_commit(WAM, "2c08af04d037d5667c02f6ddebbda9ff04581c3e")
for path, label in [(WAM_PYTHON, "WAM Python"), (WAM_CHECKPOINT, "wam_mit.pth"), (WAM_PARAMS, "params.json")]:
    require(path, label)


In [ ]:
run([
    WAM_PYTHON, SCRIPTS / "watermark_anything_video.py", "embed",
    "--source", WAM, "--checkpoint", WAM_CHECKPOINT, "--params", WAM_PARAMS,
    "--input-video", MOSE_VIDEO, "--output-video", WATERMARKED_VIDEO,
])


## Run all four inpainting methods on the frozen input

The preparation cell freezes hashes, frame count, dimensions, FPS, mask foreground counts, provenance, and any short-video padding. Every remover below consumes that frozen input and mask.


In [ ]:
ATTACK_RUN = RUN_ROOT / "attack"
if not (ATTACK_RUN / "sample.json").is_file():
    run([
        DIFFUERASER_PYTHON, SCRIPTS / "prepare_watermark_removal_sample.py", "prepare",
        "--source-video", WATERMARKED_VIDEO, "--source-mask", ALIGNED_MASK,
        "--output", ATTACK_RUN, "--watermark-method", "Watermark Anything",
        "--sample-id", "02b72b8d", "--sample-source", "MOSE fixed sample",
        "--source-kind", "mose_post_hoc",
    ])
else:
    print("Prepared sample already exists:", ATTACK_RUN / "sample.json")


In [ ]:
if not (ATTACK_RUN / "removal-pair.json").is_file():
    run([
        DIFFUERASER_PYTHON, SCRIPTS / "run_watermark_removal_pair.py",
        "--run", ATTACK_RUN, "--diffueraser", DIFFUERASER,
        "--diffueraser-python", DIFFUERASER_PYTHON,
        "--device", GPU, "--seed", "2025",
    ])
else:
    print("Both removal outputs already exist:", ATTACK_RUN / "removal-pair.json")


In [ ]:
EFFECTERASE = Path(os.environ.get("EFFECTERASE", "/mnt/data_fast/smayan/research/EffectErase"))
EFFECTERASE_PYTHON = Path(os.environ.get("EFFECTERASE_PYTHON", str(EFFECTERASE / ".venv/bin/python")))
E2FGVI = Path(os.environ.get("E2FGVI", "/mnt/data_fast/smayan/research/E2FGVI"))
E2FGVI_CHECKPOINT = Path(os.environ.get(
    "E2FGVI_CHECKPOINT", str(E2FGVI / "release_model/E2FGVI-HQ-CVPR22.pth")
))
E2FGVI_PYTHON = Path(os.environ.get("E2FGVI_PYTHON", str(DIFFUERASER_PYTHON)))

if not (ATTACK_RUN / "effecterase.json").is_file():
    effect_env = os.environ.copy(); effect_env["CUDA_VISIBLE_DEVICES"] = GPU
    run([
        EFFECTERASE_PYTHON, SCRIPTS / "run_prepared_effecterase.py",
        "--source", EFFECTERASE, "--run", ATTACK_RUN, "--device", GPU,
    ], env=effect_env)
else:
    print("EffectErase output already exists:", ATTACK_RUN / "effecterase.json")

if not (ATTACK_RUN / "e2fgvi.json").is_file():
    run([
        E2FGVI_PYTHON, SCRIPTS / "run_prepared_e2fgvi.py",
        "--source", E2FGVI, "--checkpoint", E2FGVI_CHECKPOINT,
        "--run", ATTACK_RUN, "--python", E2FGVI_PYTHON, "--device", GPU,
    ])
else:
    print("E2FGVI-HQ output already exists:", ATTACK_RUN / "e2fgvi.json")


In [ ]:
run([
    WAM_PYTHON, SCRIPTS / "watermark_anything_video.py", "detect",
    "--source", WAM, "--checkpoint", WAM_CHECKPOINT, "--params", WAM_PARAMS,
    "--gt-mask", ATTACK_RUN / "gt-mask.mp4", "--output", RUN_ROOT / "detection",
    "--case", f"original={ATTACK_RUN / 'input-watermarked.mp4'}",
    "--case", f"propainter={ATTACK_RUN / 'propainter.mp4'}",
    "--case", f"diffueraser={ATTACK_RUN / 'diffueraser.mp4'}",
    "--case", f"effecterase={ATTACK_RUN / 'effecterase.mp4'}",
    "--case", f"e2fgvi={ATTACK_RUN / 'e2fgvi.mp4'}",
])


In [ ]:
show_results(RUN_ROOT)


## Recorded one-clip four-inpainter completion

All four removers use the same frozen watermarked input and aligned binary mask for this sample. Method-specific resizing, padding, and mask dilation are recorded in the adjacent JSON rather than silently normalized.

| Inpainting method | Status | Evidence |
|---|---:|---|
| ProPainter | complete | [video](../../runs/remaining-watermarks/watermark-anything/02b72b8d/attack/propainter.mp4) | [JSON](../../runs/remaining-watermarks/watermark-anything/02b72b8d/attack/propainter.json) |
| DiffuEraser | complete | [video](../../runs/remaining-watermarks/watermark-anything/02b72b8d/attack/diffueraser.mp4) | [JSON](../../runs/remaining-watermarks/watermark-anything/02b72b8d/attack/diffueraser.json) |
| EffectErase | complete | [video](../../runs/remaining-watermarks/watermark-anything/02b72b8d/attack/effecterase.mp4) | [JSON](../../runs/remaining-watermarks/watermark-anything/02b72b8d/attack/effecterase.json) |
| E2FGVI-HQ | complete | [video](../../runs/remaining-watermarks/watermark-anything/02b72b8d/attack/e2fgvi.mp4) | [JSON](../../runs/remaining-watermarks/watermark-anything/02b72b8d/attack/e2fgvi.json) |

Watermark method: **Watermark Anything**.


In [ ]:
from pathlib import Path
import json
from IPython.display import JSON, Video, display

def find_repo_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "scripts/finalize_four_inpainter_notebooks.py").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter inside the paper-reproductions-task1 checkout")

ROOT = find_repo_root()
ATTACK = ROOT / 'runs/remaining-watermarks/watermark-anything/02b72b8d/attack'
for name in ("input-watermarked.mp4", "watermarked.mp4", "gt-mask.mp4", "propainter.mp4", "diffueraser.mp4", "effecterase.mp4", "e2fgvi.mp4"):
    path = ATTACK / name
    if path.is_file():
        print(name)
        display(Video(str(path), embed=True, html_attributes="controls muted loop"))
for name in ("sample.json", "removal-pair.json", "effecterase.json", "e2fgvi.json"):
    path = ATTACK / name
    if path.is_file():
        print(name)
        display(JSON(json.loads(path.read_text())))
